<a href="https://colab.research.google.com/github/EricXu23/bwsi-css-labs/blob/main/how_to_run_a_basic_gaussian_splatting_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Clone the Official Repository
First, we clone the repository and its submodules.

In [ ]:
!git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive

Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 1053, done.
remote: Total 1053 (delta 0), reused 0 (delta 0), pack-reused 1053 (from 1)
Receiving objects: 100% (1053/1053), 78.71 MiB | 10.31 MiB/s, done.
Resolving deltas: 100% (595/595), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core.git) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization.git) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/fused-ssim' (https://github.com/rahul-goel/fused-ssim.git) registered for path 'submodules/fused-ssim'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
remote: Enumerating objects: 3293, done.        
remote: Counting objects: 100% (322/322), done.        
remote: Compressing objects: 100% (17

2. Install Dependencies
We need to install `plyfile` and compile the custom CUDA rasterizer and KNN submodules. This step may take a few minutes.

In [ ]:
%cd gaussian-splatting
!pip install -q plyfile
!pip install ./submodules/diff-gaussian-rasterization
!pip install ./submodules/simple-knn

[Errno 2] No such file or directory: 'gaussian-splatting'
/content/gaussian-splatting
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.7 MB/s eta 0:00:00
Processing ./submodules/diff-gaussian-rasterization
  Preparing metadata (setup.py) ... done
  Created wheel for diff_gaussian_rasterization: filename=diff_gaussian_rasterization-0.0.0-cp312-cp312-linux_x86_64.whl size=3816625 sha256=70e88b4ad45213806e230715f4034bffffc00ecaa20899eb9730763f2159f772
  Stored in directory: /root/.cache/pip/wheels/01/e0/e8/f40a1cd6a1d5760cbd3036081bdad5b36c41fc11c786d4a404
Successfully built diff_gaussian_rasterization
Processing ./submodules/simple-knn
  Preparing metadata (setup.py) ... done
  Created wheel for simple_knn: filename=simple_knn-0.0.0-cp312-cp312-linux_x86_64.whl size=3555548 sha256=343e179ae8194e541619f9f78f3f2fef04c0924e45a00ef50d0195c7d3e61dfd
  Stored in directory: /root/.cache/pip/wheels/0a/f2/1b/255828ebad94ea248378281b7926639d83ce4f394f0052800d
Successfully built simple_

In [ ]:
print("Installing viewer-specific Python dependencies...")
!pip install -q uvicorn fastapi websockets dearpygui dearpygui_ext

Installing viewer-specific Python dependencies...


### 3. Download a Sample Dataset
For this example, we'll download a small sample dataset formatted for Gaussian Splatting (e.g., standard COLMAP structure).

In [ ]:
from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')
print("\nDrive mounted! You can now access your files under /content/drive/MyDrive/")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Drive mounted! You can now access your files under /content/drive/MyDrive/


### 3.5 Prepare the Dataset (COLMAP)
If your dataset is just raw images, we need to run COLMAP to extract camera poses.
*Note: Make sure your images are placed inside an `input` folder within your dataset directory (e.g., `bonsai/input/`).*

In [ ]:
!sudo apt-get install -y colmap

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libamd2 libatk-bridge2.0-0
  libatk1.0-0 libatk1.0-data libatspi2.0-0 libcamd2 libccolamd2 libceres2
  libcholmod3 libcolamd2 libcxsparse3 libdouble-conversion3 libevdev2
  libfreeimage3 libgflags2.2 libglew2.2 libgoogle-glog0v5 libgtk-3-0
  libgtk-3-bin libgtk-3-common libgudev-1.0-0 libilmbase25 libinput-bin
  libinput10 libjxr0 libmd4c0 libmetis5 libmtdev1 libopenexr25 libqt5core5a
  libqt5dbus5 libqt5gui5 libqt5network5 libqt5svg5 libqt5widgets5 libraw20
  librsvg2-common libspqr2 libsuitesparseconfig5 libwacom-bin libwacom-common
  libwacom9 libxcb-icccm4 libxcb-image0 libxcb-keysyms1 libxcb-render-util0
  libxcb-util1 libxcb-xinerama0 libxcb-xinput0 libxcb-xkb1 libxcomposite1
  libxkbcommon-x11-0 libxtst6 qt5-gtk-platformtheme qttranslations5-l10n
  session-migration
Suggested packages:
  gle

In [ ]:
import os
import shutil
from PIL import Image

# Install xvfb for headless GPU OpenGL context
os.system("sudo apt-get install -y xvfb")

dataset_path = "/content/drive/MyDrive/nerf_datasets/bonsai"
input_path = os.path.join(dataset_path, "input")

# Ensure the 'input' folder exists
if not os.path.exists(input_path):
    os.makedirs(input_path)

print(f"Organizing 'input' directory at {input_path}")
valid_exts = ['.jpg', '.jpeg', '.png', '.exr', '.tif', '.tiff']

# 1. Move any valid images from the dataset root into 'input'
for item in os.listdir(dataset_path):
    if item == "input":
        continue
    item_path = os.path.join(dataset_path, item)
    if os.path.isfile(item_path):
        if any(item.lower().endswith(ext) for ext in valid_exts):
            shutil.move(item_path, os.path.join(input_path, item))

# 2. Move any NON-image files from 'input' back to the dataset root
for item in os.listdir(input_path):
    item_path = os.path.join(input_path, item)
    if os.path.isfile(item_path):
        if not any(item.lower().endswith(ext) for ext in valid_exts):
            print(f"Moving non-image file '{item}' out of 'input' folder...")
            shutil.move(item_path, os.path.join(dataset_path, item))

# 3. Limit to 40 images
all_images = sorted([f for f in os.listdir(input_path) if any(f.lower().endswith(e) for e in valid_exts)])
MAX_IMAGES = 40

if len(all_images) > MAX_IMAGES:
    print(f"\nFound {len(all_images)} images. Reducing to {MAX_IMAGES}...")
    for img in all_images[MAX_IMAGES:]:
        shutil.move(os.path.join(input_path, img), os.path.join(dataset_path, img))

images_count = len([f for f in os.listdir(input_path) if any(f.lower().endswith(e) for e in valid_exts)])
print(f"\nReady! Found {images_count} images in the 'input' folder.")

if images_count == 0:
    print("ERROR: No images found! Please check where your images are located in Google Drive.")
else:
    # 4. Resize images to prevent RAM crash
    print("\nResizing images to max 800px to prevent Colab from running out of RAM...")
    for img_name in os.listdir(input_path):
        if any(img_name.lower().endswith(e) for e in valid_exts):
            img_path = os.path.join(input_path, img_name)
            try:
                with Image.open(img_path) as img:
                    # Only resize if larger than 800
                    if img.size[0] > 800 or img.size[1] > 800:
                        img.thumbnail((800, 800), Image.Resampling.LANCZOS)
                        img.save(img_path)
            except Exception as e:
                print(f"Error resizing {img_name}: {e}")

    # 5. Clean up ANY broken folders from previous crashes
    distorted_path = os.path.join(dataset_path, "distorted")
    sparse_path = os.path.join(dataset_path, "sparse")

    if os.path.exists(distorted_path):
        shutil.rmtree(distorted_path)
        print("Cleaned up old interrupted 'distorted' database folder.")
    if os.path.exists(sparse_path):
        shutil.rmtree(sparse_path)
        print("Cleaned up old interrupted 'sparse' folder.")

    # Set the Qt platform to offscreen for headless Colab environment
    os.environ['QT_QPA_PLATFORM'] = 'offscreen'

    # PATCH: Modify the convert.py script to use sequential_matcher instead of exhaustive_matcher
    print("\nPatching convert.py to use sequential_matcher...")
    !sed -i 's/exhaustive_matcher/sequential_matcher/g' convert.py

    # Run COLMAP with GPU using xvfb to provide a virtual display!
    print("\nStarting COLMAP extraction (GPU mode with xvfb) - Sequential Matching...")
    !xvfb-run python convert.py -s {dataset_path}

Streaming output truncated to the last 5000 lines.

iter      cost      cost_change  |gradient|   |step|    tr_ratio  tr_radius  ls_iter  iter_time  total_time
   0  1.030614e+02    0.00e+00    3.80e+02   0.00e+00   0.00e+00  1.00e+04        0    1.17e-03    2.73e-03
   1  8.089791e+01    2.22e+01    9.33e+03   1.58e+01   7.98e-01  1.27e+04        1    1.83e-03    4.60e-03
   2  7.412654e+01    6.77e+00    1.61e+04   2.98e+01   4.76e-01  1.27e+04        1    1.57e-03    6.19e-03
   3  6.445576e+01    9.67e+00    1.10e+04   2.54e+01   7.48e-01  1.44e+04        1    1.59e-03    7.80e-03
   4  6.064749e+01    3.81e+00    1.15e+04   2.55e+01   5.13e-01  1.44e+04        1    1.59e-03    9.41e-03
   5  5.625347e+01    4.39e+00    8.90e+03   2.23e+01   6.72e-01  1.51e+04        1    1.62e-03    1.11e-02
   6  5.346829e+01    2.79e+00    7.65e+03   2.03e+01   6.33e-01  1.54e+04        1    1.58e-03    1.27e-02
   7  5.119521e+01    2.27e+00    6.16e+03   1.79e+01   6.79e-01  1.61e+04        1 

### 4. Train the Model
Run the training script pointing to the extracted source images and COLMAP data.

In [ ]:
# Path to the Bonsai dataset in Google Drive
dataset_path = "/content/drive/MyDrive/nerf_datasets/bonsai"

print(f"Starting training on {dataset_path}...")
!python train.py -s {dataset_path} --eval --iterations 2000

print("\n--- Training Complete! ---")
print("To interactively view your 3D Gaussian Splatting model:")
print("1. Open the file browser on the left side of Colab.")
print("2. Navigate to the 'output' folder, find your latest run, and locate the .ply file: output/<run_id>/point_cloud/iteration_2000/point_cloud.ply")
print("3. Right-click the 'point_cloud.ply' file and select 'Download'.")
print("4. Open a web-based Gaussian Splat viewer in your browser, such as: https://antimatter15.com/splat/ or https://playcanvas.com/super-splat")
print("5. Drag and drop the downloaded .ply file into the webpage to explore your 3D model!")

Starting training on /content/drive/MyDrive/nerf_datasets/bonsai...
2026-06-25 12:53:51.732096: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Optimizing 
Output folder: ./output/fabf764a-d [25/06 12:53:56]
------------LLFF HOLD------------- [25/06 12:54:03]
Reading camera 40/40 [25/06 12:54:03]
Loading Training Cameras [25/06 12:54:04]
Loading Test Cameras [25/06 12:54:38]
Number of points at initialisation :  17831 [25/06 12:54:43]
Training progress: 100% 2000/2000 [00:57<00:00, 35.05it/s, Loss=0.0197789, Depth Loss=0.0000000]

[ITER 2000] Saving Gaussians [25/06 12:55:42]

Training complete. [25/06 12:55:44]


In [ ]:
import os
import glob

# Auto-detect the most recent training output folder that contains the model configuration
output_base = "/content/gaussian-splatting/output"
cfg_files = glob.glob(f"{output_base}/*/cfg_args")

if cfg_files:
    # Get the most recently modified training output
    latest_cfg = max(cfg_files, key=os.path.getmtime)
    model_path = os.path.dirname(latest_cfg)
    print(f"Auto-detected latest model path from train.py: {model_path}")
else:
    print("Could not auto-detect model path. Make sure training completed successfully.")
    model_path = "./output/9b346ebd-7" # Fallback

Auto-detected latest model path from train.py: /content/gaussian-splatting/output/fabf764a-d


Once the viewer starts running in the cell above, it typically launches a web server on a local port (often `6006`). Colab usually provides a clickable link for this local server, often appearing as `http://localhost:6006/` in the output, which will be proxied to a public URL. Click on that link to open the interactive viewer in a new browser tab. The server will run as long as the cell is executing, so keep it running to interact with the viewer.

### 5. Render Output Views
Since the interactive viewer is a desktop app, we will render the camera trajectories to images and display them as a video.

In [ ]:
%cd /content/gaussian-splatting
!pip install -q plyfile

# Render the images using the auto-detected trained model
# This will generate images in the output folder under 'train' and 'test' directories
!python render.py -m {model_path}

[Errno 2] No such file or directory: '/content/gaussian-splatting'
/content
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.8 MB/s eta 0:00:00
python3: can't open file '/content/render.py': [Errno 2] No such file or directory
